# Tokenizer — Building BPE from Scratch

**Video:** https://www.youtube.com/watch?v=zduSFxRajkE  
**Reference:** Karpathy's [minbpe](https://github.com/karpathy/minbpe) repo

## What this notebook builds

A byte-level BPE tokenizer, built up from first principles:

1. **Character-level tokenization** — the naive baseline. Vocab is tiny; sequences are long.
2. **Byte-level tokenization** — UTF-8 bytes. Universal (any text, any language, any file) but sequences are ~4x longer than word count.
3. **BPE from scratch** — iterative merge algorithm. Trained on a text corpus, produces a vocabulary + merge rules.
4. **Encoding** — apply merge rules to any new string.
5. **Decoding** — token IDs back to bytes back to text.
6. **Comparison against `tiktoken`** — OpenAI's production tokenizer. Correctness check.
7. **The weird behaviors** — "strawberry" has 3 tokens, non-English gets 2-4x more tokens per word, arithmetic bugs from inconsistent number splits.

## Why this matters

Tokenization is the LLM's *only* interface to text. Model quirks about spelling, counting letters, arithmetic, non-English languages — most of them trace back to tokenization decisions, not to model architecture. Understanding tokenization is how you go from "the model is bad at X" to "of course the model is bad at X, look at the token stream."

**Interview-relevance:** BPE is a top-5 topic for LLM engineer screens. "Why is 'strawberry' three tokens?" "Why can't the model count the R's?" These are the surface questions; the underlying answer is *always* tokenization.

## Rules (same as Weeks 1-4)

1. **TYPE. Do not paste.**
2. **Pause and predict** before running any new operation.
3. **Break things on purpose** — feed pathological inputs (emojis, RTL text, code snippets) and see what happens.
4. **Log what surprised you** in the session log below.

**Every time you open this notebook:** `Kernel → Restart Kernel and Run All Cells`.

## Section 1 — Character-level baseline

Start simple. Read a text corpus, build a character-level tokenizer: `stoi` and `itos` maps between chars and integer IDs.

**Why this is a bad tokenizer for LLMs:**
- Sequences are very long (one token per char)
- Vocab is tiny (~100 chars for English text), which forces the model to learn *everything* — morphology, compounds, common phrases — from scratch
- No re-use of common subword patterns

**Predict:** for a 1MB text corpus, how many characters is that vs how many "words"? Which is a bigger sequence length problem?

In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
print(''.join(chars))
print(f"vocab_size = {vocab_size}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hii there"))
print(decode(encode("hii there")))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size = 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## Section 2 — Byte-level tokenization

UTF-8 turns any string into a sequence of bytes (0-255). This is the most universal starting point for a modern tokenizer: it can represent *any* text in *any* language *without* an "unknown token" — every possible input has a valid byte representation.

**Compared to char-level:**
- English ASCII is 1 byte/char → same sequence length
- Non-English (Chinese, Arabic, emoji) is 2-4 bytes/char → sequences get *longer*
- Vocab is exactly 256, always — every possible byte value

**Why byte-level is the right foundation for BPE:** if BPE starts from bytes, it can never encounter an "unknown character" — worst case, it just emits raw byte tokens. This is what GPT-2's tokenizer does under the hood.

**Predict:** the string "héllo" — how many chars? How many UTF-8 bytes?

In [2]:
encode = lambda s: list(s.encode("utf-8"))
decode = lambda ids: bytes(ids).decode("utf-8", errors="replace")

for s in ["hii there", "héllo", "こんにちは", "🍓"]:
    ids = encode(s)
    print(f"{s!r:12s} → {len(ids):2d} bytes → {decode(ids)!r}")

'hii there'  →  9 bytes → 'hii there'
'héllo'      →  6 bytes → 'héllo'
'こんにちは'      → 15 bytes → 'こんにちは'
'🍓'          →  4 bytes → '🍓'


## Section 3 — BPE motivation + algorithm

**The problem with byte-level:** sequences are still long. "hello" is 5 tokens. A 1000-word document is ~5000 tokens. Attention is O(n²), so long sequences are expensive.

**BPE's fix:** iteratively merge the most common adjacent pair of tokens into a new single token. After 50,000 merges, common words become one token ("hello"), common phrases become one token (" the"), and rare words fall back to fragments.

### The algorithm (training)

```
1. Start with a byte-level tokenization of the entire training corpus
2. Count all adjacent pairs of tokens
3. Find the most common pair (a, b)
4. Merge every occurrence of (a, b) into a new token ID = 256 + merge_index
5. Record the merge rule: (a, b) → new_id
6. Repeat steps 2-5 until you hit the target vocab size (e.g., 500 for a toy, 50257 for GPT-2)
```

Result: a **vocabulary** (byte → ID for 0-255, plus one ID per learned merge) and a list of **merge rules** in order.

### The algorithm (encoding a new string)

```
1. Byte-encode the input string
2. Apply merges in order — for each merge rule (a, b) → new_id, scan the token sequence and replace every occurrence of (a, b) with new_id
3. Output the final token IDs
```

### The algorithm (decoding a token sequence)

```
1. For each token ID, look up its byte sequence (recursively expanding merges)
2. Concatenate all bytes, UTF-8 decode
```

**Predict:** if you train BPE with 100 merges on the tinyshakespeare corpus, what would be the FIRST merge? (Hint: what's the most common adjacent pair of bytes in English text?)

In [3]:
def get_stats(ids):
    """Count adjacent pairs. Returns dict[(a, b) -> count]."""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    """Return a new list with every occurrence of `pair` replaced by `idx`."""
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

assert get_stats([1, 2, 3, 1, 2, 5]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1, (2, 5): 1}
assert merge([1, 2, 3, 1, 2, 5], (1, 2), 99) == [99, 3, 99, 5]
print("helpers ok")

helpers ok


## Section 4 — Training the BPE tokenizer

Now put it together into a training loop. We'll train a small BPE tokenizer on the tinyshakespeare corpus with `NUM_MERGES = 500` — resulting vocab size = 256 + 500 = 756.

**What to watch:**
- The first few merges will be the most common English digraphs and short words
- Compression ratio improves as merges accumulate
- Diminishing returns — merge #499 helps much less than merge #10

**Predict:** what's the average compression ratio (chars per token) after 500 merges? Ballpark to nearest whole number.

In [4]:
NUM_MERGES = 500

tokens = list(text.encode("utf-8"))
ids = list(tokens)
merges = {}
vocab = {i: bytes([i]) for i in range(256)}

for i in range(NUM_MERGES):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    if i < 10 or (i + 1) % 100 == 0:
        print(f"merge {i:3d}: {pair} → {idx}   {vocab[idx]!r}")

print(f"\nbytes before: {len(tokens):,}")
print(f"tokens after: {len(ids):,}")
print(f"compression:  {len(tokens) / len(ids):.2f}x")

merge   0: (101, 32) → 256   b'e '
merge   1: (116, 104) → 257   b'th'
merge   2: (116, 32) → 258   b't '
merge   3: (115, 32) → 259   b's '
merge   4: (100, 32) → 260   b'd '
merge   5: (44, 32) → 261   b', '
merge   6: (111, 117) → 262   b'ou'
merge   7: (101, 114) → 263   b'er'
merge   8: (105, 110) → 264   b'in'
merge   9: (121, 32) → 265   b'y '
merge  99: (102, 97) → 355   b'fa'
merge 199: (32, 264) → 455   b' in'
merge 299: (266, 32) → 555   b'an '
merge 399: (300, 256) → 655   b'fore '
merge 499: (333, 256) → 755   b'ure '

bytes before: 1,115,394
tokens after: 490,609
compression:  2.27x


## Section 5 — Encoding new text

The training loop built two artifacts: `merges` (ordered dict of pairs → new IDs) and `vocab` (ID → bytes). Now we need a function that takes any new string and applies the merges in order.

**Key subtlety:** merges must be applied *in the same order* they were learned. Applying a later merge first would produce a different (and wrong) tokenization.

In [5]:
def encode(text):
    """Byte-level BPE encoder. Applies merges in the order they were learned."""
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
    return tokens

print(encode("hello world!"))

[104, 389, 269, 516, 108, 100, 33]


## Section 6 — Decoding

Decoding is simpler than encoding. `vocab` was built alongside `merges` during training: each new token ID maps to the byte concatenation of its two parents, all the way down to raw bytes. So decoding is just: look each ID up in `vocab`, concatenate the byte strings, UTF-8 decode.

**The one gotcha:** partial multi-byte UTF-8 sequences at token boundaries can produce invalid bytes when decoded alone. Use `errors='replace'` for robustness (invalid bytes become the Unicode replacement char, `�`).

In [6]:
def decode(ids):
    """Token IDs → bytes → UTF-8 text. `errors='replace'` handles partial UTF-8 at token boundaries."""
    return b"".join(vocab[idx] for idx in ids).decode("utf-8", errors="replace")

print(decode(encode("hello world!")))

hello world!


## Section 7 — Edge cases

Real tokenizers handle:
- **Special tokens** — sequence delimiters like `<|endoftext|>` that are single tokens, never split
- **Unicode fallback** — bytes that don't form valid UTF-8 alone (e.g., middle of a multi-byte char) still round-trip correctly
- **Pre-tokenization / regex splitting** — GPT-2 splits text on a regex FIRST (into word-ish chunks) before applying BPE, to prevent merges across word boundaries. This is why " the" and "the" and "The" can be three distinct tokens.

For a toy notebook we won't do all of these, but we should at least verify the byte round-trip works for arbitrary inputs.

In [7]:
examples = [
    "",                      # empty string
    "a",                     # single char
    "🍓",                    # emoji (4 UTF-8 bytes)
    "héllo",                 # accented char
    "こんにちは",             # Japanese
    "First Citizen:\nYou",   # newline
    " " * 20,                # runs of spaces
]

for s in examples:
    try:
        result = decode(encode(s))
        assert result == s, f"got {result!r}"
        print(f"✅ PASSED: {s!r}")
    except AssertionError as e:
        print(f"❌ FAILED on {s!r}: {e}")
    except Exception as e:
        print(f"💥 CRASHED ({type(e).__name__}) on {s!r}: {e}")

✅ PASSED: ''
✅ PASSED: 'a'
✅ PASSED: '🍓'
✅ PASSED: 'héllo'
✅ PASSED: 'こんにちは'
✅ PASSED: 'First Citizen:\nYou'
✅ PASSED: '                    '


## Section 8 — Compare against `tiktoken`

`tiktoken` is OpenAI's production tokenizer (Rust core, Python binding). It ships with pre-trained vocabularies for `cl100k_base` (GPT-4), `p50k_base` (older GPT-3.5), and others. Comparing your from-scratch BPE against tiktoken on the same text is the correctness check.

Two things to check:
1. **Compression ratio** on the same text — your 500-merge tokenizer vs GPT-4's ~100,000-token vocabulary
2. **Behavior on the classic weird cases** — "strawberry", non-English text, numbers, whitespace

**Predict:** on the tinyshakespeare corpus, GPT-4's tokenizer produces roughly how many tokens? (Yours is ~330k with 500 merges. GPT-4 has ~100k merges — how much better?)

In [8]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # GPT-4's tokenizer

s = "안녕하세요 👋 (hello in Korean!)"
tik_ids = enc.encode(s)
assert enc.decode(tik_ids) == s
print(f"tiktoken IDs for {s!r}:\n  {tik_ids}\n")

print("compression on tinyshakespeare:")
print(f"  ours (500 merges):  {len(tokens) / len(ids):.2f}x")
print(f"  tiktoken (cl100k):  {len(tokens) / len(enc.encode(text)):.2f}x")

print(f"\ntoken count on {s!r}:")
print(f"  ours:      {len(encode(s))}")
print(f"  tiktoken:  {len(tik_ids)}")

tiktoken IDs for '안녕하세요 👋 (hello in Korean!)':
  [31495, 230, 75265, 243, 92245, 62904, 233, 320, 15339, 304, 16526, 16715]

compression on tinyshakespeare:
  ours (500 merges):  2.27x
  tiktoken (cl100k):  3.70x

token count on '안녕하세요 👋 (hello in Korean!)':
  ours:      32
  tiktoken:  12


## Section 9 — The weird behaviors

Now the fun part — reproduce the classic "why does GPT do that" moments and see them fall out of tokenization.

### The strawberry problem

"How many R's are in strawberry?" GPT-4 gets this wrong because "strawberry" is a small number of tokens (2-3), and the model doesn't see individual characters — it sees token IDs. It has to LEARN counts, and character-level counts are hard to learn.

### Non-English overhead

Same content in English vs. Japanese vs. Arabic uses very different token counts. This is a real cost: OpenAI charges per token, so non-English users pay more for the same information.

### Number inconsistency

"123" and "1234" often share NO tokens. The model can't easily do arithmetic when consecutive numbers tokenize differently.

**Predict for each below** before running.

In [9]:
# 1. The strawberry problem — model sees chunks, not characters
print("=== strawberry problem ===")
for fruit in ["strawberry", "raspberry", "banana"]:
    ids = enc.encode(fruit)
    pieces = [enc.decode([i]) for i in ids]
    print(f"  {fruit:12s} → {pieces}")

# 2. Non-English overhead — same content, wildly different token counts
print("\n=== non-English overhead ===")
greetings = {
    "English":  "hello world",
    "Japanese": "こんにちは世界",
    "Arabic":   "مرحبا بالعالم",
}
for lang, s in greetings.items():
    n = len(enc.encode(s))
    print(f"  {lang:10s} {n:2d} tokens   {len(s) / n:.2f} chars/token")

# 3. Number inconsistency — consecutive integers don't share tokens
print("\n=== number inconsistency ===")
for n in [1, 12, 123, 1234, 12345, 123456]:
    print(f"  {n:>7d} → {enc.encode(str(n))}")

=== strawberry problem ===
  strawberry   → ['str', 'aw', 'berry']
  raspberry    → ['ras', 'p', 'berry']
  banana       → ['banana']

=== non-English overhead ===
  English     2 tokens   5.50 chars/token
  Japanese    4 tokens   1.75 chars/token
  Arabic     10 tokens   1.30 chars/token

=== number inconsistency ===
        1 → [16]
       12 → [717]
      123 → [4513]
     1234 → [4513, 19]
    12345 → [4513, 1774]
   123456 → [4513, 10961]


## Session log

### What clicked

- **BPE is compression with a purpose.** Without merges, byte-level sequences would eat context and blow up attention (O(n²)). But too-aggressive compression hurts too — larger vocab means more embedding parameters and rarer tokens get undertrained. The empirical sweet spot for production models sits around 100k–200k merges.
- **`errors='replace'` is the graceful-fail for partial UTF-8 at token boundaries.** Multi-byte characters can straddle a merge boundary, so a single token's byte string may not be valid UTF-8 by itself. `errors='replace'` swaps invalid bytes for `�` instead of crashing.
- **Modern tokenization is much better than char/byte on non-English and whitespace.** `cl100k_base` compresses tinyshakespeare ~3.7x vs my 500-merge tokenizer's ~2.3x, and its regex pre-split makes leading-space variants (`" the"` vs `"the"`) first-class tokens.
- **Every "GPT is bad at X" quirk traces back to token boundaries.** "How many R's in strawberry?" — the model literally never sees the R's; it sees `str aw berry` and has to *learn* how many R's each of those chunks contains. Same story with arithmetic: `123` and `1234` don't share tokens the way their digits share.

### Questions I had (and the answers I found)

**Q: Does tiktoken use BPE?**
Yes. `cl100k_base` (GPT-4) and `o200k_base` (GPT-4o) are byte-level BPE plus a regex pre-tokenizer — same algorithm as this notebook, just scaled up (~100k / ~200k merges) and written in Rust.

**Q: How is the sweet spot for vocab size found?**
Empirical tradeoff between two costs. Too small → sequences get long, attention gets expensive. Too large → embedding/unembedding tables dominate parameter count, rare tokens undertrained, softmax over the vocab expensive. Common landing spots: GPT-2 = 50,257 · GPT-4 ≈ 100,277 · Llama 3 = 128,000 · GPT-4o ≈ 200,019.

**Q: Bytes-as-tokens LLMs — has anyone actually built one?**
Yes. **ByT5** (2021), **MEGABYTE** (Meta, 2023), and **Byte Latent Transformer / BLT** (Meta, Dec 2024). All hierarchical: a small local model chunks bytes into patches, a big transformer runs on patches, then a local decoder expands back to bytes. BLT is competitive with Llama 3 8B at similar training FLOPs — with no tokenizer at all.

**Q: Why regex pre-splitting if BPE pairs bytes, not words?**
The regex is a hard barrier the merge loop can't cross. GPT-2's regex splits on word boundaries, punctuation, and leading spaces, giving chunks like `[" the", " quick", " brown"]`. BPE runs *independently on each chunk*, so pairs like `(" fox", ".")` are never counted by `get_stats` and never merged. This prevents pathological glue-together merges (`" the.", " the,"`) and makes behavior predictable — `" the"` stays `" the"` regardless of what follows.

### What surprised me

- **`errors='replace'` is what makes byte-level BPE robust.** Without it, decoding a token stream that ends mid-multibyte-character crashes. That single kwarg is the difference between a toy and something you can ship.
- **`min(stats, key=lambda p: merges.get(p, float('inf')))`** is the cleanest way to pick the next merge during encoding: any pair not in `merges` sorts to infinity and gets filtered out by the follow-up `pair not in merges` break.
- **BPE is entirely order-dependent.** `merges` has to preserve insertion order (Python 3.7+ does), and encoding must apply merges in the same order they were learned. Apply them out of order and you get a different — wrong — tokenization for the same string.